# Price model: cities A and B
Group assignment 2, deliverable 3.4. Target: `price` (nightly price in USD). Cities: **A = Los Angeles**, **B = San Francisco**. City C is not used anywhere here.

**Needs:** the table `workspace.airbnb_eda.listings_classified` and the volume `workspace.airbnb_eda.raw` created by `00_eda_airbnb` (run that notebook first), or point `SOURCE_TABLE` to the pipeline's Silver table with the columns in `pm.REQUIRED_COLUMNS`.

**What lives where.** All reusable logic is in `src/price_model.py` (feature contract, split, baseline, model, metrics, scoring, saving). This notebook only calls it and shows the results, so the first run on city C is the same function calls on another table.

**Order of work (nothing is looked at on the test set before step 7).**
1. load listings and add features (no price information is used for features);
2. split train / test per city, **before any model choice**; the split is saved;
3. fit the baseline on the training rows;
4. choose the model with cross-validation **on the training rows only**;
5. evaluate once on the test rows: MAE in USD for A, B and both, against the baseline;
6. explain and inspect the errors;
7. write predictions for **every** listing to a gold table (dashboard and Genie read it) and save the model.

Every number reported in sections 6 to 8 comes from the held-out test rows.

## 1. Setup

In [ ]:
import datetime
import importlib
import os
import shutil
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

SRC_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "src"))
assert os.path.exists(os.path.join(SRC_DIR, "price_model.py")), f"price_model.py not found in {SRC_DIR}"
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
import price_model as pm
importlib.reload(pm)

CATALOG, SCHEMA = "workspace", "airbnb_eda"
SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.listings_classified"      # one row per listing; replace with the pipeline's Silver table
CITY_ROLE = {"Los_Angeles": "A", "San_Francisco": "B"}        # city C is intentionally not listed
COLORS = {"Los_Angeles": "#1f77b4", "San_Francisco": "#ff7f0e"}
TEST_SIZE, SEED = 0.2, pm.SEED
MODEL_NAME = "price_model_v1"
MODEL_DIR = f"/Volumes/{CATALOG}/{SCHEMA}/raw/models"
GOLD = {"predictions": f"{CATALOG}.{SCHEMA}.gold_price_predictions",
        "metrics": f"{CATALOG}.{SCHEMA}.gold_model_metrics",
        "split": f"{CATALOG}.{SCHEMA}.price_model_split"}

## 2. Load listings and add features

The source table needs the columns listed in `pm.REQUIRED_COLUMNS`. A missing column is filled with missing values and a warning is printed (the pipeline must not break on a schema difference).

In [ ]:
print("required columns:", pm.REQUIRED_COLUMNS)
src = spark.table(SOURCE_TABLE).filter(F.col("city").isin(list(CITY_ROLE)))
have = [c for c in pm.REQUIRED_COLUMNS + pm.OPTIONAL_COLUMNS if c in src.columns]
data = pm.add_features(src.select(have).toPandas()).reset_index(drop=True)
data["price_status"] = pm.price_status(data)
print(data.shape)

**Listings by price status.** One row per city: all listings, listings with a valid price (used for fitting and testing), without any price, and placeholder prices (excluded from fitting and scoring).

In [ ]:
status = data.groupby(["city", "price_status"]).size().unstack("price_status").fillna(0).astype(int)
status.insert(0, "all_listings", status.sum(axis=1))
display(status.rename_axis("city").reset_index())

## 3. Split: train and test, before any model choice

Listings with a valid price are split 80 / 20, stratified by city and room type, with a fixed seed. Listings without a valid price are marked `not_labelled`: they get predictions later but are never used for fitting or scoring. The split is saved so everybody uses the same rows.

In [ ]:
data["split"] = pm.make_split(data, test_size=TEST_SIZE, seed=SEED)

train = data[data.split == "train"].reset_index(drop=True)
test = data[data.split == "test"].reset_index(drop=True)
overlap = set(zip(train.city, train.id)) & set(zip(test.city, test.id))
assert not overlap, "a listing is in both train and test"

spark.createDataFrame(data[["city", "id", "split"]].astype({"id": "int64"})).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(GOLD["split"])
print(f"split saved to {GOLD['split']}")

**Split sizes.** One row per city, one column per split, plus the share of valid-price listings in the test set.

In [ ]:
sizes = data.groupby(["city", "split"]).size().unstack("split").fillna(0).astype(int)
sizes["test_share_of_labelled_pct"] = (100 * sizes["test"] / (sizes["train"] + sizes["test"])).round(1)
display(sizes.rename_axis("city").reset_index())

**Do train and test look alike?** One row per (city, split): number of listings, quartiles of price, share of entire homes and share of monthly rentals. They should be close.

In [ ]:
chk = (data[data.split.isin(["train", "test"])].assign(entire_home=lambda x: (x.room_type == "Entire home/apt").astype(float))
       .groupby(["city", "split"]).agg(listings=("id", "size"), price_q1=("price", lambda s: s.quantile(.25)),
                                       price_median=("price", "median"), price_q3=("price", lambda s: s.quantile(.75)),
                                       pct_entire_home=("entire_home", lambda s: 100 * s.mean()),
                                       pct_monthly_rental=("is_monthly_rental", lambda s: 100 * s.mean())).round(1))
display(chk.reset_index())

## 4. Baseline

The baseline predicts, for every listing, **the median training price of listings with the same room type and number of guests** (fallback for a combination never seen in training: the room-type median, then the overall median). A second baseline also splits by stay type (short / monthly), because the EDA found that monthly listings cost about 40 % less.

In [ ]:
baseline = pm.MedianBaseline(("room_type", "accommodates")).fit(train)
baseline_stay = pm.MedianBaseline(("room_type", "accommodates", "stay_type")).fit(train)
print("baseline cells:", len(baseline.table_), "| with stay type:", len(baseline_stay.table_), "| overall median price:", round(baseline.global_, 2))

**Baseline lookup table** (training data only). One row per (room type, guests): median price and number of training listings behind it. The 15 most common combinations.

In [ ]:
display(baseline.table_.reset_index().sort_values("n_listings", ascending=False).head(15).round(2))

## 5. Features

**Feature list.** One row per feature: its type, meaning, and why it is allowed. The assertion below fails if any column from the EDA leakage review (prices and price-derived columns, quote details, availability, reviews, identifiers) is among the features.

In [ ]:
pm.assert_no_leakage()
print("no leaking column among the", len(pm.FEATURES), "features")
display(pd.DataFrame(pm.FEATURE_DOC, columns=["feature", "type", "meaning", "why it is allowed"]))

**Missing values in the features** (training rows). One row per feature: % missing. Boosting handles them natively; ridge imputes the median and adds a missing-indicator.

In [ ]:
miss = (100 * train[pm.FEATURES].isna().mean()).round(2).rename("pct_missing").rename_axis("feature").reset_index()
display(miss.sort_values("pct_missing", ascending=False))

## 6. Choosing the model (training rows only)

Five candidates (two ridge regressions, three boosting settings) and the baseline are compared with 5-fold cross-validation **inside the training rows**. The test rows are not touched. Metric: MAE in USD per night on the held-out fold; `cv_mae_std` = spread over the 5 folds.

In [ ]:
rows = []
cv = pm.cross_validate(lambda: pm.MedianBaseline(("room_type", "accommodates")), train)
rows.append({"candidate": "baseline: room type x guests", "kind": "baseline", **cv})
for name, (kind, params) in pm.CANDIDATES.items():
    cv = pm.cross_validate(lambda kind=kind, params=params: pm.PriceModel(kind, params), train)
    rows.append({"candidate": name, "kind": kind, **cv})
selection = pd.DataFrame(rows).sort_values("cv_mae_mean").reset_index(drop=True)
display(selection.round(2))

best_name = selection[selection.kind != "baseline"].iloc[0]["candidate"]
best_kind, best_params = pm.CANDIDATES[best_name]
print("chosen:", best_name, best_params)

**Final fit.** The chosen candidate is fitted once on all training rows.

In [ ]:
model = pm.PriceModel(best_kind, best_params).fit(train)
print(model.describe()["kind"], "| training listings:", model.n_train_, "| property types kept:", len(model.top_types_), "+ Other")

## 7. Evaluation on the test rows

Predictions for **all** listings are produced once (train, test and unlabelled), in the layout of the gold table. Metrics below use the test rows only.

In [ ]:
scored = pm.score_listings(data, model, baseline, baseline_stay, split=data["split"], model_name=MODEL_NAME,
                           scored_at=f"{datetime.datetime.now():%Y-%m-%d %H:%M:%S}")
test_scored = scored[scored.split == "test"].reset_index(drop=True)
PRED = {"baseline": "baseline_price", "baseline + stay type": "baseline_stay_type_price", "model": "predicted_price"}
scopes = pm.city_scopes(test_scored, CITY_ROLE)
results = pm.metrics_table(test_scored, PRED, scopes)
print("test listings:", len(test_scored))

**Test-set error.** One row per (scope, model). `mae` = mean absolute error in USD per night (the assignment's metric); `median_ae` = typical error; `rmse` = punishes large errors; `mape_pct` = mean absolute % error; `mean_error` = predicted minus actual (negative = under-prices on average); `r2_log` = R-squared on log price.

In [ ]:
display(results.round(2))

**Improvement over the baseline.** One row per scope: MAE of each, and how much lower the model's MAE is than the baseline's (positive = better).

In [ ]:
wide = results.pivot(index="scope", columns="model", values="mae")[["baseline", "baseline + stay type", "model"]]
wide["model_vs_baseline_pct"] = (100 * (1 - wide["model"] / wide["baseline"])).round(1)
wide["model_vs_baseline_stay_pct"] = (100 * (1 - wide["model"] / wide["baseline + stay type"])).round(1)
display(wide.round(2).rename_axis("scope").reset_index())

ax = wide[["baseline", "baseline + stay type", "model"]].plot.bar(figsize=(9, 4), rot=0, color=["#bbbbbb", "#888888", "#2ca02c"])
for c in ax.containers:
    ax.bar_label(c, fmt="$%.0f", fontsize=8)
ax.set(title="Test-set MAE: the model against the baselines", xlabel="", ylabel="MAE, USD per night")
plt.tight_layout(); plt.show()

**Error by stay type.** One row per (city, stay type): test listings and MAE of each model.

In [ ]:
display(pm.mae_by(test_scored, ["city", "stay_type"], PRED).round(2))

**Error by room type.** One row per (city, room type).

In [ ]:
display(pm.mae_by(test_scored, ["city", "room_type"], PRED).round(2))

**Error by number of guests.** One row per (city, guest group).

In [ ]:
test_scored["guests"] = pd.cut(test_scored["accommodates"], [0, 1, 2, 4, 6, 8, 100], labels=["1", "2", "3-4", "5-6", "7-8", "9+"])
display(pm.mae_by(test_scored, ["city", "guests"], PRED).round(2))

## 8. Visualisations and error analysis (test rows)

**Predicted vs actual price.** Each dot is a test listing, both axes on a log scale; the dashed line is a perfect prediction.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharex=True, sharey=True)
for ax, city in zip(axes, CITY_ROLE):
    d = test_scored[(test_scored.city == city) & test_scored.error_usd.notna()]
    ax.scatter(d.actual_price, d.predicted_price, s=6, alpha=0.3, color=COLORS[city])
    lim = [max(10, d.actual_price.min()), d.actual_price.max()]
    ax.plot(lim, lim, "k--", lw=1)
    ax.set(xscale="log", yscale="log", title=f"{CITY_ROLE[city]}: {city}", xlabel="actual price, USD per night", ylabel="predicted price, USD per night")
plt.tight_layout(); plt.show()

**Size of the errors.** Left: ratio predicted / actual price on a log scale (0 = exact, +0.69 = predicted twice the actual). Right: absolute error in USD, clipped at $500.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for city in CITY_ROLE:
    d = test_scored[(test_scored.city == city) & test_scored.error_usd.notna()]
    axes[0].hist(np.log(d.predicted_price / d.actual_price).clip(-1.5, 1.5), bins=60, alpha=0.6, color=COLORS[city], density=True, label=city)
    axes[1].hist(d.abs_error_usd.clip(upper=500), bins=60, alpha=0.6, color=COLORS[city], density=True, label=city)
axes[0].axvline(0, color="black", lw=1); axes[0].legend()
axes[0].set(title="log(predicted / actual)", xlabel="log ratio", ylabel="density")
axes[1].set(title="Absolute error (clipped at $500)", xlabel="USD per night", ylabel="density")
plt.tight_layout(); plt.show()

**Calibration.** Test listings are cut into 10 groups by predicted price; for each group the average predicted and the average actual price. Close columns mean the model is right on average at every price level.

In [ ]:
d = test_scored[test_scored.error_usd.notna()].copy()
d["predicted_decile"] = pd.qcut(d.predicted_price, 10, labels=False, duplicates="drop") + 1
cal = d.groupby("predicted_decile").agg(listings=("id", "size"), mean_predicted=("predicted_price", "mean"),
                                        mean_actual=("actual_price", "mean"), mean_abs_error=("abs_error_usd", "mean")).round(1)
display(cal.reset_index())

**Error by neighbourhood.** One row per (city, neighbourhood) with at least 15 test listings: MAE and mean error (negative = under-priced by the model). The 8 neighbourhoods with the largest MAE per city.

In [ ]:
for city in CITY_ROLE:
    g = (test_scored[(test_scored.city == city) & test_scored.error_usd.notna()].groupby("neighbourhood_cleansed")
         .agg(test_listings=("id", "size"), median_actual_price=("actual_price", "median"), mae=("abs_error_usd", "mean"),
              mean_error=("error_usd", "mean")).query("test_listings >= 15").sort_values("mae", ascending=False).head(8).round(1))
    print(f"{CITY_ROLE[city]}: {city}")
    display(g.reset_index())

**Listings the model under-prices most** (predicted far below actual): the 12 test listings with the most negative error.

In [ ]:
COLS = ["city", "id", "room_type", "property_type", "accommodates", "bedrooms", "minimum_nights", "actual_price", "predicted_price", "error_usd", "pct_error"]
display(test_scored.nsmallest(12, "error_usd")[COLS])

**Listings the model over-prices most** (predicted far above actual): the 12 test listings with the largest positive error.

In [ ]:
display(test_scored.nlargest(12, "error_usd")[COLS])

## 9. What the model uses

**Permutation importance** (test rows). Each group of features is shuffled across listings and the model's MAE is recomputed: `mae_increase_usd` = how many dollars worse the predictions get without that information (5 repeats; `mae_increase_std` = spread). Location features are shuffled together because they overlap.

In [ ]:
eval_rows = data[(data.split == "test")].reset_index(drop=True)
importance = pm.permutation_importance_mae(model, eval_rows, n_repeats=5, seed=SEED)
display(importance.round(2))

ax = importance.iloc[::-1].plot.barh(x="feature_group", y="mae_increase_usd", legend=False, figsize=(8, 4), color="#2ca02c")
ax.set(title="Which information the model relies on", xlabel="increase in test MAE when the group is shuffled, USD per night", ylabel="")
plt.tight_layout(); plt.show()

## 10. Predictions for every listing (gold table) and saved model

The gold table has one row per listing (train, test and not labelled): actual and predicted price, both baselines, errors and the split. Column comments are set so Genie understands the columns. A second table holds the test metrics for the dashboard.

In [ ]:
gold = scored.astype(object).where(scored.notna(), None)
(spark.createDataFrame(gold, schema=pm.GOLD_DDL).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(GOLD["predictions"]))
for col, comment in pm.GOLD_COMMENTS.items():
    spark.sql(f"ALTER TABLE {GOLD['predictions']} ALTER COLUMN {col} COMMENT '{comment}'")
spark.sql(f"COMMENT ON TABLE {GOLD['predictions']} IS 'Model price prediction for every listing with actual price, baselines and errors; evaluate on rows where split = test'")

metrics_out = results.assign(split="test", model_name=MODEL_NAME, evaluated_at=f"{datetime.datetime.now():%Y-%m-%d %H:%M:%S}")
spark.createDataFrame(metrics_out).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(GOLD["metrics"])
print("written:", GOLD["predictions"], "and", GOLD["metrics"])

**Check.** One row per (city, split): listings in the gold table against listings in the source data. They must be equal.

In [ ]:
in_gold = spark.table(GOLD["predictions"]).groupBy("city", "split").agg(F.count("*").alias("rows_in_gold_table")).toPandas()
expected = data.groupby(["city", "split"]).size().rename("rows_in_data").reset_index()
check = expected.merge(in_gold, on=["city", "split"], how="outer")
assert (check.rows_in_data == check.rows_in_gold_table).all(), "gold table does not contain every listing"
display(check)

**Save the model** (with both baselines and a JSON description) to the volume, so the run on city C can load it without refitting.

In [ ]:
meta = {"model_name": MODEL_NAME, "trained_at": f"{datetime.datetime.now():%Y-%m-%d %H:%M:%S}", "source_table": SOURCE_TABLE,
        "cities": list(CITY_ROLE), "test_size": TEST_SIZE, "chosen_candidate": best_name,
        "test_mae": {f"{r.scope} | {r.model}": round(r.mae, 2) for r in results.itertuples()}}
tmp_path = f"/tmp/{MODEL_NAME}.joblib"
pm.save_model(tmp_path, model, baseline, baseline_stay, meta)
os.makedirs(MODEL_DIR, exist_ok=True)
for ext in (".joblib", ".json"):
    shutil.copyfile(f"/tmp/{MODEL_NAME}{ext}", f"{MODEL_DIR}/{MODEL_NAME}{ext}")
print("saved to", f"{MODEL_DIR}/{MODEL_NAME}.joblib")

## 11. Summary and how to reuse

In [ ]:
print(f"Model: {best_name} (fitted on log price, converted back to USD), {model.n_train_:,} training listings, {len(pm.FEATURES)} features")
print(f"Test set: {len(test_scored):,} listings, split {int((1 - TEST_SIZE) * 100)}/{int(TEST_SIZE * 100)} per city\n")
print(f"{'scope':<22}{'baseline MAE':>14}{'baseline+stay':>15}{'model MAE':>12}{'model vs baseline':>20}")
for scope in scopes:
    r = results[results.scope == scope].set_index("model")["mae"]
    print(f"{scope:<22}{r['baseline']:>13.2f}${r['baseline + stay type']:>14.2f}${r['model']:>11.2f}${100 * (1 - r['model'] / r['baseline']):>18.1f} %")

**Reuse for the next steps**
- **Dashboard and Genie:** read `gold_price_predictions` (one row per listing, `split = 'test'` for honest error numbers) and `gold_model_metrics`.
- **City C (week 2):** the same functions on the new city's table, without refitting:
```python
saved = pm.load_model(f"{MODEL_DIR}/{MODEL_NAME}.joblib")
c_data = pm.add_features(spark.table("<silver table of city C>").select(...).toPandas())
c_scored = pm.score_listings(c_data, saved["model"], saved["baseline"], saved["baseline_stay_type"], split_label="new_data")
pm.metrics_table(c_scored, {"baseline": "baseline_price", "model": "predicted_price"}, {"C": c_scored.price_status == "valid"})
```
  Missing columns are filled with missing values (with a printed warning), unseen property types and neighbourhoods are tolerated; to change the model for city C, edit `FEATURES` or `CANDIDATES` in `src/price_model.py` and rerun this notebook.